<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Textverarbeitung mit LangChain
</b></font> </br></p>


---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.model_config import BASELINE, WORKER

In [ ]:
#@title 🛠️ Installationen { display-mode: "form" }
install_packages([('markdown-pdf', 'markdown_pdf'), 'pypdf'])

# 1 | Aufgabenbereiche Sprachverarbeitung
---


Large Language Models (LLMs) wie GPT übernehmen eine Vielzahl von Aufgaben rund um die Verarbeitung natürlicher Sprache. Neben bekannten Anwendungsfeldern wie Textklassifizierung, Textzusammenfassung und Textgenerierung bieten sie auch Unterstützung bei Analyse, Transformation, Strukturierung und Wissensgewinnung aus Texten. Die folgende Übersicht zeigt typische Aufgabenbereiche im Überblick.

| **Aufgabenbereiche**                        | **Aufgabe**                      | **Beschreibung**                                                                     |
| ------------------------------------ | -------------------------------- | ------------------------------------------------------------------------------------ |
| 🧠 **Analyse & Klassifikation**      | **Textklassifizierung**          | Einordnung eines Textes in vordefinierte Klassen (z. B. Spam/kein Spam).             |
|                                      | Sentimentanalyse                 | Einschätzung der Stimmung (positiv/negativ/neutral).                                 |
|                                      | Themenklassifikation             | Zuordnung von Texten zu Themen oder Kategorien.                                      |
|                                      | **Named Entity Recognition (NER)**   | Erkennung von Namen, Orten, Organisationen usw. im Text.                             |
|                                      | Relationsextraktion              | Erkennen logischer oder semantischer Beziehungen zwischen Entitäten.                 |
|                                      | Bias- oder Hate-Speech-Erkennung | Klassifikation diskriminierender oder voreingenommener Sprache.                      |
| 📝 **Generierung & Transformation**  | **Textgenerierung**              | Erzeugen von Texten auf Basis von Prompts (z. B. Geschichten, Antworten, Code).      |
|                                      | **Textzusammenfassung**          | Komprimierung eines Textes unter Erhalt der Kernaussage (extraktiv oder abstraktiv). |
|                                      | Paraphrasierung                  | Umformulierung bei gleichbleibendem Sinn.                                            |
|                                      | Stiltransfer                     | Umwandlung des Sprachstils (z. B. sachlich ↔ kreativ).                               |
|                                      | Textvereinfachung                | Vereinfachung komplexer Formulierungen.                                              |
|                                      | Sprachübersetzung                | Übersetzung zwischen Sprachen.                                                       |
| 🧩 **Ergänzung & Vervollständigung** | Autovervollständigung            | Vorhersage des nächsten Wortes oder Satzes.                                          |
|                                      | Lückentext-Ergänzung             | Einfügen sinnvoller Wörter/Sätze in Lücken.                                          |
|                                      | **Story- oder Textfortsetzung**      | Kreative Weiterführung eines gegebenen Textes.                                       |
| 📚 **Informationsgewinnung**         | Schlüsselbegriffextraktion       | Extraktion zentraler Begriffe aus einem Text.                                        |
|                                      | Themenextraktion                 | Identifikation der zentralen Themen eines Textes.                                    |
|                                      | Frage-Antwort (QA)               | Beantwortung von Fragen zu einem Text (extraktiv oder generativ).                    |
|                                      | Faktenprüfung                    | Bewertung der Richtigkeit von Aussagen.                                              |
|                                      | **Retrieval-Augmented Generation**   | Kombination von Textgenerierung mit Wissensabfragen aus externen Quellen.            |
| 🛡️ **Sicherheit & Strukturierung**  | Plagiatserkennung                | Erkennung von Ähnlichkeiten mit bestehenden Texten.                                  |
|                                      | Anonymisierung                   | Entfernen oder Maskieren sensibler Daten.                                            |
|                                      | **Gliederungserstellung**            | Erstellung strukturierter Inhaltsverzeichnisse.                                      |
|                                      | Formatumwandlung                 | Konvertierung von Texten in bestimmte Formate (z. B. E-Mail, Bericht, Code).         |


# 2 | Modellaufbau
---


Im vorliegenden Code wird ein Chatbot-Modell mit Chat-History schrittweise aufgebaut. Der Modellaufbau ist modular und kann als `Bauplan` für nachfolgende Konversationsmodelle verwendet werden.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Verarbeitungskette</font> </br></p>

diagram = """
%%{init: {'theme':'light'}}%%
flowchart LR
    Input["Input<br/>(Message)"] --> Prompt

    subgraph Chain
        Prompt["Prompt<br/>(Template)"]
        Model["Model<br/>(LLM)"]
        Response["Response<br/>(Parser)"]
        Prompt --> Model --> Response
    end

    Response --> Result["Result"]
"""
mermaid(diagram)



Diese strukturierte Aufteilung macht das Modell robust, wartbar und erweiterbar für verschiedene Anwendungsfälle im Bereich der Konversations-KI.



In [ ]:
# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers.string import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# Konstanten definieren
system_prompt = """
Du bist ein kompetenter KI-Assistent mit breitem Fachwissen.

Deine Antworten sind:
- Klar strukturiert und mit Markdown formatiert
- Praxisorientiert und direkt umsetzbar
- Basierend auf aktuellem Kenntnisstand
- Mit passenden Beispielen versehen
- In verständlicher Sprache formuliert

Bei deiner Arbeit:
- Analysierst du Fragen sorgfältig
- Gibst präzise und relevante Antworten
- Erkennst den Kontext der Anfrage
- Bietest bei Bedarf weiterführende Informationen
- Bleibst sachlich und neutral

Formatiere alle Antworten in Markdown für optimale Lesbarkeit.
"""

In [ ]:
# Chat-Prompt für Klassifizierung (kein MessagesPlaceholder — History wird in chat() manuell aufgebaut)
prompt = ChatPromptTemplate([
        ("system", "{system_prompt}"),
        ("user", "{user_input}")
])

# LLM definieren (Kurznotation: "provider:model")
llm = init_chat_model(BASELINE)

# OutputParser definieren
parser = StrOutputParser()

# Chain erstellen
chain = prompt | llm | parser

In [ ]:
# Funktionen definieren
def chat(chat_history: list, user_input: str) -> list:
    """Chat mit manueller Listen-Memory: Kontext bauen → invoke → anhängen."""
    context  = [SystemMessage(content=system_prompt)] + chat_history + [HumanMessage(content=user_input)]
    response = llm.invoke(context)

    # Ausgabe der Ergebnisse
    mprint("### 🧑‍🦱 Mensch:")
    mprint(user_input)

    mprint("### 🤖 KI:")
    mprint(response.content)

    # Memory-Management - Erweiterung der Historie
    chat_history.extend([
        HumanMessage(content=user_input),
        response
    ])

    return chat_history

In [ ]:
# Hauptprogramm

# Historie wird initialisiert — SystemMessage wird in chat() ergänzt
chat_history = []

# Liste mit user_input
user_input_list = [
    "Mein Name ist Max",
    "Warum ist der Himmel blau?",
    "Und warum ist er manchmal rot?",
    "Wie ist mein Name?"
]

# Chain-Aufruf in einer Schleife
for user_input in user_input_list:
    chat_history = chat(chat_history, user_input)

In [ ]:
chat_history

In [ ]:
len(chat_history)

<p><font color='black' size="5">
Neue Konversation starten
</font></p>

In [ ]:
# Hauptprogramm
chat_history = []  # SystemMessage wird in chat() ergänzt
user_input = "Erstelle eine Tabelle der fünf bevölkerungsreichsten Ländern mit Bevölkerung und BIP."
chat_history = chat(chat_history, user_input)

In [ ]:
len(chat_history)

# 3 | Textgenerierung
---

Die Generierung von Text gehört zu den häufigsten Anwendungsfällen für **Large Language Models (LLMs)**. Anstatt eine interaktive Chatumgebung zu verwenden, erfolgt die Textgenerierung durch gezielte **Eingabeaufforderungen (Prompts)**, die an **OpenAI** gesendet werden. Das Modell verarbeitet diese Anfragen und liefert darauf basierend den generierten Text.  

<p><font color='black' size="5">
Muster zur Textgenerierung
</font></p>

Bei der grundlegenden Textgenerierung gibt es verschiedene **Prompt-Muster (Prompting-Methoden)**, die sich je nach Detailgrad der bereitgestellten Informationen unterscheiden. Diese Muster beeinflussen, wie das **Large Language Model (LLM)** den gewünschten Text erzeugt.  

Im Folgenden werden verschiedene dieser Muster untersucht und analysiert, wie sie sich auf die Qualität und Struktur der generierten Inhalte auswirken.

* Zero-Shot-Prompt
* One-Shot-Prompt
* Few-Shot-Prompt

<p><font color='black' size="5">
Zero-Shot Textgenerierung
</font></p>

Ein Zero-Shot-Prompt zur Texterstellung ist eine Technik, bei der ein Sprachmodell ohne vorherige Anpassung oder spezielles Training auf eine bestimmte Aufgabe direkt mit einer einzigen Eingabeaufforderung genutzt wird. Um diesen Ansatz erfolgreich anzuwenden, ist es entscheidend, einen präzisen und ausführlichen Prompt zu formulieren, der genau beschreibt, welche Art von Inhalt generiert werden soll. Dabei sollten Stil, Struktur und relevante Details oder Einschränkungen klar angegeben werden. Beispielsweise kann für eine geschäftliche E-Mail der gewünschte Tonfall (formell oder informell), die wesentlichen Inhalte (wie Termin, Thema, Teilnehmer) sowie eine Handlungsaufforderung spezifiziert werden. Da das Modell allein auf die im Prompt enthaltenen Informationen angewiesen ist, sollte das gewünschte Ergebnis möglichst eindeutig formuliert sein. Diese Methode ist flexibel einsetzbar und erlaubt die Erstellung vielfältiger Textarten, ohne dass eine vorherige Modellanpassung erforderlich ist.  

Das folgende Beispiel demonstriert eine Zero-Shot-Eingabeaufforderung: Es werden verschiedene Anforderungen gestellt und Informationen zu einem Studierenden bereitgestellt, jedoch ohne eine explizite Vorlage für das Sprachmodell vorzugeben.

**Es wird der bereits erstellte ChatBot verwendet und leicht angepasst.**

In [ ]:
# Konstanten definieren
system_prompt = """
Du bist ein hilfreicher KI-Assistent, der bei der Erstellung von Empfehlungsschreiben hilft.
"""

In [ ]:
user_input = load_prompt(
    "https://github.com/ralf-42/GenAI/blob/main/05_prompt/m03_empfehlung_zero_shot.md", mode="S"
)

In [ ]:
chat_history = []
chat_history = chat(chat_history, user_input)

<p><font color='black' size="5">
One-Shot Textgenerierung
</font></p>

Ein One-Shot-Prompt zur Textgenerierung ist eine Methode, bei der ein Sprachmodell mit einer einzigen, detaillierten Eingabe angewiesen wird, auf deren Basis es einen zusammenhängenden Text erstellt. Um diese Technik wirkungsvoll einzusetzen, sollte der Prompt klar und präzise formuliert sein und alle relevanten Informationen sowie den gewünschten Kontext enthalten. Dabei ist es ratsam, Stil, Ton und spezifische inhaltliche Aspekte genau zu definieren.  

Möchte man beispielsweise eine atmosphärische Beschreibung einer Küstenstadt generieren lassen, sollten wesentliche Merkmale wie Tageszeit, Stimmung und bestimmte visuelle oder emotionale Eindrücke explizit benannt werden. Eine solche detaillierte Vorgabe erleichtert es dem Modell, die Anforderungen zu erfassen und passgenaue Inhalte zu erstellen.  

Nachdem der Prompt formuliert wurde, kann er direkt in das Textgenerierungstool eingegeben werden. Falls das Ergebnis nicht vollständig den Erwartungen entspricht, lässt sich der Prompt gezielt anpassen, um die Qualität und Relevanz der Ausgabe weiter zu optimieren.

In [ ]:
user_input = load_prompt(
    "https://github.com/ralf-42/GenAI/blob/main/05_prompt/m03_empfehlung_one_shot.md", mode="S"
)

In [ ]:
chat_history = []
chat_history = chat(chat_history, user_input)

<p><font color='black' size="5">
Synthetische Daten generieren
</font></p>

LLMs (Large Language Models) eignen sich zur Erzeugung synthetischer Daten, was besonders nützlich für Testszenarien ist, die realistische Informationen oder eine breite demografische Vielfalt erfordern. Diese Modelle können beispielsweise detaillierte Biografien für verschiedene Berufsgruppen generieren, wodurch realitätsnahe Daten für Simulationen, die Entwicklung von Testalgorithmen oder das Training anderer KI-Systeme bereitgestellt werden.  

Ein möglicher Anwendungsfall wäre die Erstellung synthetischer Biografien für Berufsgruppen wie Softwareentwickler, Kinderkrankenschwestern, Finanzanalysten, Naturwissenschaftslehrer an Universitäten oder Marketingmanager. Jede dieser Biografien könnte individuelle Karrierewege, Qualifikationen und berufliche Erfahrungen umfassen, um vielseitige und praxisnahe Tests zu ermöglichen. Auf diese Weise tragen synthetische Daten zur Verbesserung der Systemleistung bei, ohne auf echte personenbezogene Informationen zurückzugreifen.

In [ ]:
# Konstanten definieren
system_prompt = """
Du bist ein Assistent, der synthetische Daten für eine Person in dem von Ihnen angegebenen Berufsfeld generiert. Erstelle eine kurze Biografie der Person, die nicht länger als 5 Sätze ist. Erwähne die Berufsbezeichnung nicht ausdrücklich. Halte dich streng an die Anweisungen.
"""

In [ ]:
# Hauptprogramm
user_input = "Softwareentwickler"

chat_history = []
chat_history = chat(chat_history, user_input)

Nun wird eine Liste mit mehreren zufällig generierten Biografien erstellt.

In [ ]:
personas = [
"Luftballon-Verkäufer",
"Löwen-Krankenschwester",
"Talkmaster",
"Hochschullehrer Wirtschaftswissenschaften",
"Zirkus-Zauberer"
]

In [ ]:
# ── Sonstige Bibliotheken ─────────────────────────────────────────────────────
from tqdm import tqdm

# Verwenden Sie tqdm, um den Fortschrittsbalken anzuzeigen
chat_history = []

for i in tqdm(range(5), desc="Generating Careers"):
    user_input = personas[i]
    chat_history = chat(chat_history, user_input)

# 4 | Textklassifizierung
---

Große Sprachmodelle (LLMs) haben die Textklassifizierung grundlegend verändert, indem sie einen flexibleren und effizienteren Ansatz als herkömmliche maschinelle Lernmethoden bieten. Während traditionelle Verfahren umfangreiche, manuell annotierte Datensätze benötigten, um Modelle zu trainieren, bieten LLMs eine zeitsparende Alternative.  

Durch die **Few-Shot Klassifizierung**, die mit beschriftete Beispiele funktioniert, können LLMs Texte allein auf Basis weniger natürlichsprachlichen Beispiele kategorisieren. Diese Fähigkeit beruht auf dem umfassenden Wissen, das die Modelle während ihres Trainings mit großen Textkorpora erworben haben.  

Dieser Ansatz reduziert den Aufwand für die Einrichtung einer Textklassifizierung erheblich und ermöglicht eine schnelle Anpassung an neue Anforderungen. Dadurch eignen sich LLMs für verschiedene Anwendungsfälle, darunter **Stimmungsanalyse, Themenkategorisierung, Spam-Erkennung und Inhaltsmoderation**.  

Im folgenden Code werden SMS-Texte analysiert. Anschließend erfolgt deren Klassifizierung mithilfe eines LLMs.

Im folgenden Programm erfolgt eine Klassifizierung der SMS-Texte.


In [ ]:
# Konstanten definieren
system_prompt = """
Du bist ein Experte für die Klassifikation von SMS-Nachrichten.

Klassifiziere SMS-Nachrichten als:
- spam: Werbung, Gewinnspiele, kostenpflichtige Dienste, unerwünschte Nachrichten
- ham: Normale Kommunikation, persönliche Nachrichten
- sonstiges: Wenn keine klare Zuordnung möglich ist

Wichtig: Antworte NUR mit einem Wort: spam, ham oder sonstiges

Beachte die Beispiele für die Klassifikation:
- spam: Werbung, Gewinnspiele, kostenpflichtige Dienste
- ham: Normale Kommunikation, persönliche Nachrichten
"""

# Beispiel-SMS für das Training
train_sms = [
            {
                "sms": "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's",
                "classification": "spam"
            },
            {
                "sms": "FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv",
                "classification": "spam"
            },
            {
                "sms": "WINNER!! As a valued network customer you have been selected to receive a £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.",
                "classification": "spam"
            },
            {
                "sms": "Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",
                "classification": "ham"
            },
            {
                "sms": "Ok lar... Joking wif u oni...",
                "classification": "ham"
            },
            {
                "sms": "U dun say so early hor... U c already then say...",
                "classification": "ham"
            },
            {
                "sms": "Hey, what time is the meeting tomorrow?",
                "classification": "ham"
            },
            {
                "sms": "I'll be home in 10 minutes",
                "classification": "ham"
            },
            {
                "sms": "Call me when you get this message",
                "classification": "ham"
            },
            {
                "sms": "FREE! Claim your prize now! Click: bit.ly/prize123",
                "classification": "spam"
            }
]

test_sms = [
    "WINNER!! As a valued customer you have won £1000 prize!",
    "Hey, what time is the meeting tomorrow?",
    "Free entry! Win an iPhone now! Click here:",
    "I'll be home in 10 minutes"
]

In [ ]:
# Modell definieren (Kurznotation: "provider:model")
llm = init_chat_model(BASELINE)

In [ ]:
# Parser erstellen
parser = StrOutputParser()

In [ ]:
# Verkettung
chain = prompt | llm | parser

In [ ]:
def create_training_examples(train_sms):
    """ Erstellt Trainingsbeispiele aus den Beispiel-SMS """
    examples = []
    for item in train_sms:
        examples.append(f"{item['classification']}\t{item['sms']}")
    return "\n".join(examples)

def classify(system_prompt, train_sms, test_sms):
    """ Klassifiziert eine einzelne SMS-Nachricht """
    # Erstelle Prompt mit Trainingsbeispielen
    full_prompt = f"""
    Klassifiziere die folgende SMS basierend auf diesen Beispielen:

    {create_training_examples(train_sms)}

    Gebe als Antwort NUR "spam" oder "ham" zurück.
    Wenn keine Entscheidung möglich ist dann "sonstiges".
    Gebe keine Begründung.
    Verwende keine Spiegelstriche.

    Hier ist die SMS:
    {test_sms}
    """
    parameter = {
        'system_prompt': system_prompt,
        'user_input': full_prompt
    }
    response = chain.invoke(parameter)

    return response.strip()

In [ ]:
# Hauptprogramm
mprint("## ✨ Klassifiziere Test-SMS:")
mprint("---")

# SMS-Klassifizierung
for i, sms in enumerate(test_sms, 1):
    response = classify(system_prompt, train_sms, sms)
    mprint(f"**Test SMS** #{i}:")
    mprint(sms)
    mprint(f"**Klassifizierung:** {response}")
    print()

<p><font color='black' size="5">
Namen klassifizieren
</font></p>

Die Analyse von Wörtern in Texten stellt für künstliche Intelligenz eine interessante Herausforderung dar, besonders bei der Zero-Shot Klassifizierung ohne Trainingsdaten. Hierbei muss das Modell spontan entscheiden, ob ein Wort ein Name oder ein reguläres Wort ist. Im folgenden Text soll getestet werden, ob eine generative KI in der Lage ist, Wörter korrekt in die Kategorien "Name" oder "Nicht-Name" einzuordnen, ohne zuvor mit ähnlichen Beispielen trainiert worden zu sein. Die zu analysierenden Wörter enthalten sowohl gewöhnliche Begriffe als auch verschiedene Arten von Namen wie Vornamen, Nachnamen oder Firmennamen. Die zentrale Frage lautet: Kann das Modell durch sein grundlegendes Sprachverständnis diese Unterscheidung treffen?

Nun wird ein Large Language Model (LLM) definiert, das den Text analysiert.

Letztendlich wird eine einzelne Eingabeaufforderung genutzt, um diese Namen zu identifizieren und zu kategorisieren.

In [ ]:
# ── Typen ─────────────────────────────────────────────────────────────────────
from typing import List

# ── Pydantic ──────────────────────────────────────────────────────────────────
from pydantic import BaseModel, Field

In [ ]:
# Pydantic-Modell für die Ausgabe definieren
class Names(BaseModel):
    names: List[str] = Field(description="Eine Liste der extrahierten Namen")

In [ ]:
# System-Prompt
system_prompt_structured = """
Du bist ein präziser Namensextraktor.

**Aufgabe:** Extrahiere ausschließlich die Namen aus dem gegebenen Text und gib sie als JSON-Liste zurück.

**Was zu extrahieren ist:**
- Personennamen (Vor- und Nachnamen)
- Unternehmensnamen
- Produktnamen

**Wichtig:**
- Gib NUR die extrahierten Namen im JSON-Format aus. Die Struktur wird durch das bereitgestellte Schema vorgegeben.
- Formuliere den Originaltext NICHT um.
- Füge KEINE weiteren Informationen hinzu.
"""

# Die Chat-Prompt definieren mit Variablen für system_prompt, history und user_input
prompt_structured = ChatPromptTemplate([
    ("system", "{system_prompt}"),
    ("user", "{user_input}")
])

# LLM definieren (Kurznotation: "provider:model")
llm_structured = init_chat_model(BASELINE)

# Kette mit structured_output erstellen
chain_structured = prompt_structured | llm_structured.with_structured_output(Names)

Schließlich rufen wir die Kette auf und geben die extrahierten Namen aus. Die Ausgabe ist nun ein `Names`-Objekt, das eine Liste von Strings enthält, welche die extrahierten Namen sind.

In [ ]:
# Hauptprogramm mit structured_output

user_input = """
Anna und Sarah sind seit dem Studium befreundet. Nach ihrem Abschluss hatten sie ein Vorstellungsgespräch bei Frau Weber für eine Stelle bei der Technovision GmbH, deren Hauptprodukt Futurtech heißt und von der Programmiererin Maria Schmidt entwickelt wurde. Nach dem erfolgreichen Gespräch trafen sie sich mit ihren Freunden Rafael und Aida zum Feiern. An ihrem ersten Arbeitstag lernten sie drei weitere Mitarbeiter kennen: Richard, Lisa und Pia. Später kam noch Matthias Müller zum Team dazu.
"""

# Namen extrahieren und anzeigen
parameter_structured = {
    'system_prompt': system_prompt_structured,
    'user_input': user_input
}

response_structured = chain_structured.invoke(parameter_structured)

mprint("## 📛 Gefundene Namen (structured output):")
mprint("---")
# Die Antwort ist jetzt ein Pydantic-Objekt, wir greifen auf das 'names'-Attribut zu
for name in response_structured.names:
    print(name)

# 5 | Textzusammenfassung
---

Große Sprachmodelle (LLMs) wie GPT-x fassen Texte zusammen, indem sie zentrale Inhalte extrahieren und kompakt wiedergeben. Sie erfassen den Kontext sowie semantische Strukturen und erstellen eine prägnante Version, die die Hauptaussagen erhält. Durch ihre Fähigkeit, verschiedene Textarten zu analysieren – von Fachartikeln bis hin zu narrativen Texten - liefern sie verständliche und relevante Zusammenfassungen. Ihre Flexibilität in Bezug auf Länge und inhaltliche Schwerpunkte macht sie zu einem effizienten Werkzeug für die schnelle Informationsverarbeitung.

<p><font color='black' size="5">
Einzelnes PDF zusammenfassen
</font></p>

Zunächst wird die Zusammenfassung einer einzelnen PDF-Datei betrachtet. LangChang nutzt spezielle Dokumentlader, um verschiedene Dateiformate wie PDFs zu verarbeiten. Für unterschiedliche Datentypen stehen spezifische Lader zur Verfügung. Im folgenden Code wird eine PDF-Datei geladen und mithilfe einer allgemeinen Systemaufforderung analysiert und zusammengefasst.

Zunächst wird das PDF-Dokument *Attention Is All You Need* über den **„PyPDFLoader“** von der angegebenen URL (**https://arxiv.org/pdf/1706.03762**) geladen. Anschliessend wird der Text an das LLM übergeben. Schließlich erfolgt die Ausgabe des zusammengefassten Inhalts im **Markdown-Format**, um die ursprüngliche Formatierung beizubehalten.

In [ ]:
# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# Chat-Prompt-Template erstellen
prompt = ChatPromptTemplate([
    ("system", "Du bist ein hilfreicher Assistent, der präzise Zusammenfassungen erstellt."),
    ("user", """Schreibe eine kurze Zusammenfassung der präsentierten Informationen.
    Schreibe die Zusammenfassung auf Deutsch.

{user_input}

ZUSAMMENFASSUNG:""")
])

# Modell definieren (Kurznotation: "provider:model")
llm = init_chat_model(BASELINE)

# Parser definieren
parser = StrOutputParser()

# Kette erstellen
chain = prompt | llm | parser


In [ ]:
# Funktionen definieren
def load_document(url: str):
    """Lädt ein PDF-Dokument von einer URL und splittet es in Chunks"""
    loader = PyPDFLoader(url)
    return loader.load_and_split()

In [ ]:
# Hauptprogramm
url = "https://arxiv.org/pdf/1706.03762"

# Dokument laden
docs = load_document(url)

# Alle Chunks zu einem Text zusammenfügen
text = " ".join([doc.page_content for doc in docs])

# Zusammenfassung generieren
response = chain.invoke({"user_input": text})

# Ausgabe
mprint("## ✨ Zusammenfassung:")
mprint("---")
mprint(response)

<p><font color='black' size="5">
Mehrere PDFs zusammenfassen
</font></p>

Diese vier wissenschaftlichen Arbeiten haben wesentliche Fortschritte im Bereich der generativen KI und der Verarbeitung natürlicher Sprache (NLP) ermöglicht.  

1. **"Attention Is All You Need" (Vaswani et al., 2017)** führte die **Transformer-Architektur** ein, die sich als grundlegendes Modell für moderne NLP-Systeme etabliert hat. Der Verzicht auf rekurrente Strukturen und die Einführung des Selbstaufmerksamkeitsmechanismus ermöglichten effizientere und leistungsfähigere Modelle wie GPT und BERT.  

2. **"BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding" (Devlin et al., 2018)** revolutionierte die Kontextverarbeitung in NLP-Modellen durch **bidirektionales Training**, das es ermöglichte, sowohl vorhergehende als auch nachfolgende Wörter gleichzeitig zu analysieren. Dadurch erzielte BERT erhebliche Verbesserungen bei zahlreichen NLP-Aufgaben.  

3. **"Language Models are Few-Shot Learners" (Brown et al., 2020)** untersuchte das Potenzial **großskalierter Transformer-Modelle** am Beispiel von GPT-3. Die Studie zeigte, dass die Leistung solcher Modelle durch Skalierung verbessert wird und sie auch ohne aufgabenspezifisches Training effektive Ergebnisse liefern können.  

4. **"Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer" (Raffel et al., 2019)** präsentierte **T5 (Text-to-Text Transfer Transformer)**, ein Modell, das alle NLP-Aufgaben in ein einheitliches Text-zu-Text-Format überführt. Dadurch wurde das Transferlernen zwischen verschiedenen Aufgaben vereinfacht.  

Diese Arbeiten bilden zusammen die Grundlage für viele moderne NLP-Modelle und deren Anwendungen in generativer KI.

In [ ]:
# Hauptprogramm
urls = [
    "https://arxiv.org/pdf/1706.03762",
    "https://arxiv.org/pdf/1810.04805",
    "https://arxiv.org/pdf/2005.14165",
    "https://arxiv.org/pdf/1910.10683"
]

# Liste für Sammlung der Summaries initialisieren
summaries = []

for url in urls:
    # Dokumente laden und als String formatieren
    print(f"Verarbeite Artikel: {url}")
    docs = load_document(url)
    text = " ".join([doc.page_content for doc in docs])

    # LLM aufrufen und Zusammenfassung erstellen
    response = chain.invoke({"user_input": text})

    # Zusammenfassungen sammeln
    summaries.append(response)

    # Zusammenfassung ausgeben
    mprint("## ✨ Zusammenfassung:")
    mprint("---")
    mprint(response)
    print()

Nachdem die einzelnen Artikel zusammengefasst wurden, folgt die Erstellung einer übergreifenden Übersicht durch die Kombination dieser Zusammenfassungen. Hierfür werden die Texte zunächst zu einer langen Zeichenfolge zusammengeführt.  

Das Endergebnis wird im **Markdown-Format** ausgegeben, um eine klare Struktur und Lesbarkeit zu gewährleisten. Auf diese Weise entsteht eine prägnante, aber inhaltlich umfassende Synthese der Originalartikel.

In [ ]:
# Gesamtzusammenfassung erstellen
gesamt_text = " ".join(summaries)
response = chain.invoke({"user_input": gesamt_text})

mprint("## ✨ Gesamt-Zusammenfassung:")
mprint("---")
mprint(response)

# 6 | LLM schreibt ein Buch
---

**Dieser** Abschnitt zeigt, wie ein Large Language Model (LLM) beim Schreiben eines Buches genutzt werden kann. Da ein Buch eine komplexe, langfristige Aufgabe darstellt, erfolgt der Prozess schrittweise.  

Zunächst wird ein **Thema** gewählt, auf dessen Basis das LLM einen Titel und eine kurze **Zusammenfassung** erstellt. Diese Zusammenfassung dient als inhaltlicher Leitfaden. Anschließend wird ein **Inhaltsverzeichnis** generiert, das die Kapitel und Abschnitte strukturiert.  

Jedes **Kapitel** wird in separaten Sitzungen mit dem LLM ausgearbeitet, wobei die zuvor erstellte Gliederung zur Orientierung dient. Dieser iterative Ansatz gewährleistet Konsistenz und ermöglicht eine detaillierte Ausarbeitung.  

Durch eine methodische Vorgehensweise verwandelt sich eine anfängliche Idee in ein gut strukturiertes Buch. Für kreative Ergebnisse wird ein Sprachmodell mit einer Temperatur von 0,7 verwendet.

In [ ]:
# Konstanten definieren
system_prompt = """
Du bist ein kreativer Autor und Schriftsteller.

Deine Texte sind:
- Fesselnd und gut strukturiert
- Reich an Details und Beschreibungen
- Konsistent im Stil und der Erzählweise
- Mit klarer Handlung und Charakterentwicklung
- In ansprechendem Deutsch verfasst

Bei deinem Schreiben:
- Entwickelst du überzeugende Charaktere
- Erschaffst atmosphärische Szenen
- Behältst den roten Faden bei
- Verwendest kreative Sprachbilder
- Achtest auf Spannung und Dynamik

Halte dich streng an die Anweisungen im user-prompt.
"""

In [ ]:
# EINFACHES ChatPromptTemplate ohne Historie
prompt = ChatPromptTemplate([
    ("system", "{system_prompt}"),
    ("user", "{user_input}")
])

In [ ]:
# Modell initialisieren
llm = init_chat_model(WORKER)

In [ ]:
# Output Parser
parser = StrOutputParser()

In [ ]:
# Chain-Erstellung
chain = prompt | llm | parser

Einfache Hilfsfunktionen werden implementiert, um das LLM mit einer benutzerdefinierten Systemaufforderung zu befragen. Diese Aufforderung informiert das Modell darüber, dass es beim Schreiben eines Buches assistiert.


<p><font color='black' size="5">
Titel, Synopse, Inhaltsverzeichnis
</font></p>

Für dieses Buch kann der Benutzer das Thema über die Variable `SUBJECT` festlegen. Anschließend wird das LLM angewiesen, einen zufälligen Titel basierend auf diesem Thema zu generieren. Dabei wird darauf geachtet, dass die Eingabeaufforderung präzise formuliert ist, um zu vermeiden, dass das LLM zusätzliche Einleitungen wie „Hier ist ein zufälliger Titel“ hinzufügt.

<p><font color='black' size="5">
Titel
</font></p>

In [ ]:
num_chapters = 6                        # Anzahl der Kapitel
subject = "Fantasy Thriller"            # Genre/Themenbereich
user_input = f"""
Erstelle einen zufälligen Titel für ein Buch zum Thema {subject}. Gebe nur den Titel als Antwort zurück.
"""

In [ ]:
parameter = {}
parameter['system_prompt'] = system_prompt
parameter['user_input'] = user_input
response_titel = chain.invoke(parameter)

mprint("## ✨ Titel:")
mprint("---")
mprint(response_titel)

<p><font color='black' size="5">
Synopsis
</font></p>

Nachdem der Titel festgelegt wurde, kann nun eine zufällige Zusammenfassung des Buches generiert werden. Diese gibt eine erste inhaltliche Orientierung und dient als Grundlage für die weitere Strukturierung.

In [ ]:
user_input = f"""
Gebe mir eine Inhaltsangabe für ein Buch "{response_titel}" zum Thema "{subject}"
Gebe nur die Inhaltsangabe zurück, keinen zusätzlichen Text.
"""

In [ ]:
parameter = {}
parameter['system_prompt'] = system_prompt
parameter['user_input'] = user_input
response_synopsis = chain.invoke(parameter)

mprint("## ✨ Synopsis:")
mprint("---")
mprint(response_synopsis)

<p><font color='black' size="5">
Inhaltsverzeichnis
</font></p>

Nun wird das Inhaltsverzeichnis erstellt, wobei alle zuvor generierten Informationen berücksichtigt werden. Ein spezifisches Format wird vorgegeben, um eine klare Struktur zu gewährleisten. Obwohl die Kapitelnummern leicht ableitbar wären, werden sie explizit angefordert, da das LLM sie ohnehin bereitstellen möchte. Der Ansatz erleichtert eine konsistente Struktur und ermöglicht es, die Nummerierung später gezielt zu entfernen, falls erforderlich.

In [ ]:
user_input = f"""
Gebe mir ein Inhaltsverzeichnis für ein Buch mit dem Titel "{response_titel}"
für ein Buch zum Thema {subject}. Die Buchzusammenfassung lautet "{response_synopsis}".
Gebe das Inhaltsverzeichnis als Liste von Kapitelüberschriften zurück.
Erstelle nicht mehr als {num_chapters} Kapitel.
Trenne Kapitelnummer und Kapitelüberschrift mit einem Pipe-Zeichen '|'.
Gebe nur die Kapitelnamen zurück, keinen zusätzlichen Text.
"""

In [ ]:
parameter = {}
parameter['system_prompt'] = system_prompt
parameter['user_input'] = user_input
response_toc = chain.invoke(parameter)

mprint("## ✨ Inhaltsverzeichnis:")
mprint("---")
print(response_toc)


<p><font color='black' size="5">
Erstellen der Kapitel des Buches
</font></p>


Nun wird eine Funktion erstellt, die den Text eines Kapitels generiert. Damit das LLM über ausreichend Kontext verfügt, werden die Zusammenfassung, das Inhaltsverzeichnis und die entsprechende Kapitelnummer übergeben.

In [ ]:
# Aufbereitung Inhaltsverzeichnis, Titel & initialisieren String für Buchinhalt
titel_liste = [zeile for zeile in response_toc.strip().split('\n')]
response_titel = response_titel.replace('"', '')
book_content = f"# {response_titel} \n ---\n\n ## Inhaltsverzeichnis \n --- \n\n {response_toc}"

In [ ]:
# ChatPromptTemplate für die iterative Erstellung der Kapitel
book_prompt = ChatPromptTemplate([
    ("system", system_prompt),
    ("user", """
Schreibe Kapitel mit dem Titel {chapter_title} für ein Buch
mit dem Titel {title} zum Thema {subject}.
Die Buchzusammenfassung lautet: {synopsis}
Das Inhaltsverzeichnis lautet: {toc}

Gebe nur den Kapiteltext zurück, keine Kapitelüberschrift, keinen Kapiteltitel,
keine Kapitelnummer, keinen zusätzlichen Text.
""")
])

In [ ]:
# Iteration über alle Kapitel mit automatischer Nummerierung
book_chain = book_prompt | llm | parser
mprint("### Kapitel erstellen")
mprint("---")

for chapter_titel in titel_liste:
    # Kapitelinhalt generieren
    response_content = book_chain.invoke({
        "chapter_title": chapter_titel,
        "title": response_titel,
        "subject": subject,
        "synopsis": response_synopsis,
        "toc": response_toc
    })

    # Hinzufügen des Kapitels zum Buchinhalt
    book_content += f"\n## {chapter_titel}\n ---\n {response_content}\n"
    mprint(f"✓ {chapter_titel}")

mprint("\n---")
mprint(f"### ✨ Buch '{response_titel}' erfolgreich erstellt!")

<p><font color='black' size="5">
PDF-Datei erstellen
</font></p>

In [ ]:
#@markdown   <p><font size="4" color='green'>  Style Angaben</font> </br></p>

%%writefile style.css
/* 1. Schriftstil Arial für das gesamte Dokument */
body {
    font-family: Arial, Helvetica, sans-serif;
    color: #222;
    line-height: 1.4;
    margin: 0;
    padding: 20px;
}

/* 2. Überschriften in Arial und Farbe #40c */
h1, h2 {
    font-family: Arial, Helvetica, sans-serif;
    color: #40c;
}

h1 {
    text-align: center;
    margin-bottom: 40px;
}

/* 3. Seitenumbruch VOR jedem h2-Element (Unterkapitel) */
h2 {
    /* Unterstützung für verschiedene Browser-Engines */
    page-break-before: always; /* Klassischer Standard */
    break-before: page;        /* Moderner Standard */
    margin-top: 50px;
}

In [ ]:
# ── Dokumentverarbeitung ──────────────────────────────────────────────────────
from markdown_pdf import MarkdownPdf, Section

pdf = MarkdownPdf()

pdf.meta["title"] = response_titel
pdf.meta["author"] = WORKER

with open("style.css") as f:
    css = f.read()

pdf.add_section(Section(book_content), user_css=css)
pdf.save("my_book.pdf")

Das generierte Buch steht nun zum Download bereit.

In [ ]:
# ── Colab ─────────────────────────────────────────────────────────────────────
from google.colab import files
files.download("my_book.pdf")

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Bei einer Aufgabe, bei der man festhängt, kann zum Beispiel Gemini in Google Colab helfen: Fehlermeldungen besser verstehen, Ideen für Teilschritte entwickeln oder Code-Varianten prüfen.
> Wichtig ist: Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, GenAI-Apps selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

**Grundlagen**

Klassifiziere eine einzelne Produktbewertung als Positiv, Neutral oder Negativ mit einem klaren Prompt.

**✅ Erledigt wenn:** Das Modell gibt einen der drei Werte aus — `positiv`, `neutral` oder `negativ` — und nennt die entscheidende Textstelle als kurze Begründung.

In [ ]:
# Grundlagen: Einzelne Produktbewertung klassifizieren
# Startpunkt: init_chat_model() + ChatPromptTemplate

bewertung = "Das Produkt ist super, aber der Versand hat ewig gedauert."

# 1. Prompt: Klassifiziere als positiv / neutral / negativ + Begründung
# 2. Modell aufrufen
# 3. Ausgabe prüfen
# ...

**Aufbau**

Führe eine aspektbasierte Sentiment-Analyse für mindestens drei Bewertungen durch (z.B. Preis, Qualität, Service getrennt bewerten).

**✅ Erledigt wenn:** Die Ausgabe enthält für jede der drei Bewertungen mindestens zwei Aspekte mit je einem Sentiment-Wert.

In [ ]:
# Aufbau: Aspektbasierte Analyse für 3 Bewertungen
# Aspekte: Preis, Qualität, Service

bewertungen = [
    "...",  # Bewertung 1
    "...",  # Bewertung 2
    "...",  # Bewertung 3
]

# Für jede Bewertung Aspekte extrahieren und bewerten
# ...

**Vertiefung**

Erzeuge eine strukturierte Ausgabe (JSON) für jede Bewertung und vergleiche zwei verschiedene Prompt-Formulierungen in ihrer Klassifikationsqualität.

**✅ Erledigt wenn:** `json.loads(response)` wirft keinen Fehler; der Unterschied zwischen beiden Prompt-Varianten ist tabellarisch dokumentiert.

In [ ]:
# Vertiefung: JSON-Ausgabe + Prompt-Vergleich
# Option A: with_structured_output() (Startpunkt: Kapitel M06)
# Option B: JSON im System-Prompt erzwingen, dann json.loads() prüfen

# ── Stdlib ────────────────────────────────────────────────────────────────────
import json

# Prompt-Variante 1 und 2 definieren
# Beide aufrufen, JSON-Validität prüfen, Unterschied dokumentieren
# ...

# B | Dokumente zum Weiterlesen
---




📚 Ergänzende Artikel aus der Kurs-Dokumentation:

- [Transformer Architektur](https://ralf-42.github.io/GenAI/03-grundlagen/transformer.html)
- [Tokenizing & Chunking](https://ralf-42.github.io/GenAI/03-grundlagen/tokenizing-chunking.html)
- [Embeddings](https://ralf-42.github.io/GenAI/03-grundlagen/embeddings.html)
- [Modellauswahl](https://ralf-42.github.io/GenAI/04-modelle-provider/modellauswahl.html)
- [Strukturierte Ausgaben](https://ralf-42.github.io/GenAI/03-grundlagen/strukturierte-ausgaben.html)